In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ahmedhany323/invoice-data/dataset_ubdate_cloud/sample_2000_train.json
/kaggle/input/datasets/ahmedhany323/invoice-data/dataset_ubdate_cloud/sample_2000_val.json
/kaggle/input/datasets/ahmedhany323/invoice-data/dataset_ubdate_cloud/sample_2000_test.json


In [2]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 105.4 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
from datasets import load_dataset
ds = load_dataset("json", data_files={
    "train": "/kaggle/input/datasets/ahmedhany323/invoice-data/dataset_ubdate_cloud/sample_2000_train.json",
    "val": "/kaggle/input/datasets/ahmedhany323/invoice-data/dataset_ubdate_cloud/sample_2000_val.json", 
    "test": "/kaggle/input/datasets/ahmedhany323/invoice-data/dataset_ubdate_cloud/sample_2000_test.json"
})

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
import json
import re
from datetime import datetime
# label schema
LABELS = ["O", "B-VENDOR", "I-VENDOR", "B-DATE", "I-DATE", "B-AMOUNT", "I-AMOUNT"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}


def find_span(text: str, target: str) -> tuple[int, int] | None:
    """Find the start/end character positions of target in text. Returns None if not found."""
    target_lower = target.lower().strip()
    text_lower = text.lower()
    idx = text_lower.find(target_lower)
    if idx == -1:
        return None
    return (idx, idx + len(target))


def find_date_span(text: str, date_iso: str) -> tuple[int, int] | None:
    """Find date in text, handling many OCR formats including space-separated digits."""
    try:
        dt = datetime.fromisoformat(date_iso)
    except ValueError:
        return None
    
    text_lower = text.lower()
    
    # String-format attempts (full month names etc.)
    string_formats = [
        dt.strftime("%B %d %Y"),            # "March 16 1972"
        dt.strftime("%B %d, %Y"),           # "March 16, 1972"
        dt.strftime("%b %d %Y"),            # "Mar 16 1972"
        dt.strftime("%b %d, %Y"),           # "Mar 16, 1972"
        dt.strftime("%d %B %Y"),            # "16 March 1972"
        dt.strftime("%d %b %Y"),            # "16 Mar 1972"
        dt.strftime("%Y-%m-%d"),            # "1972-03-16"
    ]
    for fmt in string_formats:
        idx = text_lower.find(fmt.lower())
        if idx != -1:
            return (idx, idx + len(fmt))
    
    # Numeric formats with various separators
    y, m, d = dt.year, dt.month, dt.day
    y2 = y % 100  # 2-digit year
    
    numeric_patterns = [
        # MM/DD/YYYY, MM-DD-YYYY, MM.DD.YYYY
        f"{m:02d}/{d:02d}/{y}", f"{m}/{d}/{y}",
        f"{m:02d}-{d:02d}-{y}", f"{m}-{d}-{y}",
        f"{m:02d}.{d:02d}.{y}", f"{m}.{d}.{y}",
        # MM/DD/YY with 2-digit year
        f"{m:02d}/{d:02d}/{y2:02d}", f"{m}/{d}/{y2}",
        f"{m:02d}-{d:02d}-{y2:02d}", f"{m}-{d}-{y2}",
        # DD/MM/YYYY (European)
        f"{d:02d}/{m:02d}/{y}", f"{d}/{m}/{y}",
        f"{d:02d}-{m:02d}-{y}", f"{d}-{m}-{y}",
        # Space-separated (OCR artifact): "D M YYYY" or "D M YY"
        f"{d} {m} {y}", f"{d} {m} {y2}",
        f"{d:02d} {m:02d} {y}", f"{d:02d} {m:02d} {y2:02d}",
        # Reversed space-separated: "M D YYYY"
        f"{m} {d} {y}", f"{m} {d} {y2}",
        # Compact: YYYYMMDD, DDMMYY
        f"{y}{m:02d}{d:02d}",
        f"{d:02d}{m:02d}{y2:02d}",
        f"{d:02d}{m:02d}{y}",
    ]
    for pat in numeric_patterns:
        idx = text.find(pat)
        if idx != -1:
            return (idx, idx + len(pat))
    
    return None


def find_amount_span(text: str, amount: float) -> tuple[int, int] | None:
    """Find amount span, handling OCR artifacts like space-separated digits, comma thousands, etc."""
    # Skip trivially-small amounts that would cause false matches
    if amount < 10:
        return None
    
    int_amt = int(amount)
    has_decimal = amount != int_amt
    
    # Build list of format variants in priority order (most specific first)
    variants = []
    
    if has_decimal:
        # "1234.56" with decimal
        variants.append(f"{amount:.2f}")
        # "1,234.56" with comma thousands + decimal
        variants.append(f"{int_amt:,}.{int(round((amount - int_amt) * 100)):02d}")
        # "1234 56" space-separated dollars/cents (OCR artifact)
        cents = int(round((amount - int_amt) * 100))
        variants.append(f"{int_amt} {cents:02d}")
    
    # Integer representations
    variants.append(f"{int_amt:,}")     # "1,234" comma thousands
    variants.append(str(int_amt))       # "1234" plain
    
    # Space-grouped thousands (OCR-style): "1 234" or "20 437"
    int_str = str(int_amt)
    if len(int_str) > 3:
        # Insert space every 3 digits from the right
        reversed_digits = int_str[::-1]
        grouped = ' '.join(reversed_digits[i:i+3] for i in range(0, len(reversed_digits), 3))
        space_grouped = grouped[::-1]
        variants.append(space_grouped)
    
    # Try each variant
    for variant in variants:
        idx = text.find(variant)
        if idx != -1:
            return (idx, idx + len(variant))
    
    return None

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_label(example):
    """Convert one JSON record to tokenized input with BIO labels."""
    text = example["text"]
    
    # Find character spans for each field
    spans = []
    
    vendor_span = find_span(text, example["vendor"])
    if vendor_span:
        spans.append((vendor_span[0], vendor_span[1], "VENDOR"))
    
    date_span = find_date_span(text, example["date"])
    if date_span:
        spans.append((date_span[0], date_span[1], "DATE"))
    
    amount_span = find_amount_span(text, example["total_amount"])
    if amount_span:
        spans.append((amount_span[0], amount_span[1], "AMOUNT"))
    
    # Tokenize with offsets so we can map back to character positions
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
    )
    
    # Initialize all labels to "O"
    labels = [label2id["O"]] * len(tokenized["input_ids"])
    
    # Mark special tokens ([CLS], [SEP]) as -100 (ignored in loss)
    for i, (start, end) in enumerate(tokenized["offset_mapping"]):
        if start == 0 and end == 0:  # special token
            labels[i] = -100
    
    # Assign BIO labels based on spans
    for span_start, span_end, field_type in spans:
        first_token = True
        for i, (tok_start, tok_end) in enumerate(tokenized["offset_mapping"]):
            if tok_start == 0 and tok_end == 0:  # skip special tokens
                continue
            # Does this token overlap with our span?
            if tok_start >= span_start and tok_end <= span_end:
                if first_token:
                    labels[i] = label2id[f"B-{field_type}"]
                    first_token = False
                else:
                    labels[i] = label2id[f"I-{field_type}"]
    
    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")  # not needed after labeling
    return tokenized

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
tokenized_ds = ds.map(tokenize_and_label, remove_columns=ds["train"].column_names)

# Sanity check — look at one example

print(tokenized_ds["train"][0])

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'input_ids': [101, 1021, 11460, 19739, 24657, 2531, 12376, 26224, 20958, 2487, 5170, 6384, 5100, 2531, 6643, 9103, 3927, 2047, 2259, 1050, 1061, 2531, 16576, 2035, 2937, 4063, 12621, 18505, 5482, 2236, 9517, 2233, 2385, 3285, 3680, 14433, 3585, 1996, 9098, 2820, 13963, 1047, 2395, 22064, 2899, 1040, 1039, 6203, 14433, 2720, 12621, 24340, 2038, 2356, 2033, 2000, 2830, 1996, 10837, 8001, 4638, 2000, 2899, 2118, 1999, 1996, 3815, 1997, 2322, 4724, 2581, 2753, 25664, 17850, 2015, 1052, 4830, 19665, 3187, 2000, 2720, 12621, 24340, 6289, 22851, 17539, 5170, 6384, 5100, 1051, 16409, 7295, 21382, 23632, 4700, 22794, 2290, 24665, 15937, 2047, 2259, 1050, 1061, 1042, 1050, 1061, 1018, 1042, 11253, 5167, 1015, 1016, 2213, 1050, 16666, 21084, 1052, 1049, 3815, 12731, 3367, 1999, 2615, 1016, 9388, 2321, 3285, 1015, 1022, 2034, 2120, 2103, 2924, 13412, 2683, 2322, 4724, 2581, 2753, 21373, 2395, 2012, 7063, 13642, 2047, 2259, 1050, 1061, 3477, 2023, 3815, 2531, 12376, 26224, 20958, 2487, 2050, 3599,

In [6]:
def debug_example(example, tokenizer):
    """Print the tokens + their labels to eyeball the result."""
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
    labels = example["labels"]
    
    print(f"{'TOKEN':<20} {'LABEL':<12}")
    print("-" * 35)
    for tok, lab_id in zip(tokens, labels):
        lab = id2label[lab_id] if lab_id != -100 else "IGNORE"
        print(f"{tok:<20} {lab:<12}")

# Run on one training example
debug_example(tokenized_ds["train"][0], tokenizer)

TOKEN                LABEL       
-----------------------------------
[CLS]                IGNORE      
7                    O           
mg                   O           
gu                   O           
##aldi               O           
100                  O           
##50                 O           
##49                 O           
##42                 O           
##1                  O           
philip               B-VENDOR    
morris               I-VENDOR    
incorporated         I-VENDOR    
100                  O           
pa                   O           
##ju                 O           
avenue               O           
new                  O           
york                 O           
n                    O           
y                    O           
100                  O           
##17                 O           
all                  O           
##ian                O           
##der                O           
holt                 O           
##iman      

In [7]:
def count_labeled(ds_split):
    stats = {"VENDOR": 0, "DATE": 0, "AMOUNT": 0, "total": 0}
    for ex in ds_split:
        stats["total"] += 1
        labels = [l for l in ex["labels"] if l != -100]
        if label2id["B-VENDOR"] in labels:
            stats["VENDOR"] += 1
        if label2id["B-DATE"] in labels:
            stats["DATE"] += 1
        if label2id["B-AMOUNT"] in labels:
            stats["AMOUNT"] += 1
    return stats

print("Train:", count_labeled(tokenized_ds["train"]))
print("Val:  ", count_labeled(tokenized_ds["val"]))

Train: {'VENDOR': 1155, 'DATE': 922, 'AMOUNT': 1319, 'total': 1400}
Val:   {'VENDOR': 257, 'DATE': 201, 'AMOUNT': 283, 'total': 300}


In [8]:
!pip install seqeval evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0eee9eadec647466d2a70c40f2b2e5efd13a4f6b753b5504bf7d3290ea7fb105
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [11]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import evaluate

MODEL_NAME = "distilbert-base-uncased"
LABELS = ["O", "B-VENDOR", "I-VENDOR", "B-DATE", "I-DATE", "B-AMOUNT", "I-AMOUNT"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

# ═══════════════════════════════════════
# Data collator — handles dynamic padding per batch
# ═══════════════════════════════════════
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
)

# Metrics — per-field F1 using seqeval
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (-100) and convert ids → label strings
    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
        # Per-entity scores
        "vendor_f1": results.get("VENDOR", {}).get("f1", 0.0),
        "date_f1": results.get("DATE", {}).get("f1", 0.0),
        "amount_f1": results.get("AMOUNT", {}).get("f1", 0.0),
    }


training_args = TrainingArguments(
    output_dir="./invoice-extractor",
    learning_rate=5e-5,           # ← up from 2e-5
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,          # ← up from 5
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=True,
    warmup_ratio=0.1,             # ← new: helps stability
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["val"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save final model
trainer.save_model("./invoice-extractor-final")
tokenizer.save_pretrained("./invoice-extractor-final")
print("✅ Model saved")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args,

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Vendor F1,Date F1,Amount F1
1,No log,0.319287,0.000000,0.000000,0.000000,0.964655,0.000000,0.000000,0.000000
2,1.060171,0.174859,0.386221,0.249663,0.303279,0.974175,0.339552,0.466321,0.026846
3,0.205834,0.143736,0.418465,0.470985,0.443175,0.975505,0.386023,0.645570,0.320000
4,0.128032,0.130415,0.490566,0.456140,0.472727,0.978698,0.415094,0.668235,0.362105
5,0.093829,0.129474,0.533137,0.488529,0.509859,0.980177,0.444444,0.706173,0.417355
6,0.067658,0.141126,0.524439,0.535762,0.530040,0.979674,0.432323,0.740741,0.455342
7,0.052475,0.144590,0.521940,0.609987,0.562539,0.979216,0.469534,0.753880,0.505017
8,0.039816,0.147577,0.550318,0.582996,0.566186,0.980310,0.481696,0.754023,0.500000
9,0.039816,0.154996,0.552529,0.574899,0.563492,0.979925,0.476000,0.765376,0.485166
10,0.032152,0.154640,0.554172,0.600540,0.576425,0.979659,0.478011,0.774194,0.517888


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved


In [12]:
test_output = trainer.predict(tokenized_ds["test"])
print("Test metrics:")
for k, v in test_output.metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test metrics:
  test_loss: 0.1466
  test_precision: 0.5530
  test_recall: 0.6160
  test_f1: 0.5828
  test_accuracy: 0.9813
  test_vendor_f1: 0.4981
  test_date_f1: 0.7692
  test_amount_f1: 0.5189
  test_runtime: 3.1451
  test_samples_per_second: 95.3880
  test_steps_per_second: 1.5900


In [13]:
import shutil
import os
from pathlib import Path

# Delete all per-epoch checkpoints, keep only the final saved model
ckpt_dir = Path("/kaggle/working/invoice-extractor")
if ckpt_dir.exists():
    for item in ckpt_dir.iterdir():
        if item.name.startswith("checkpoint-"):
            print(f"Removing {item}")
            shutil.rmtree(item)

# Check what's left
!du -sh /kaggle/working/*

Removing /kaggle/working/invoice-extractor/checkpoint-132
Removing /kaggle/working/invoice-extractor/checkpoint-264
Removing /kaggle/working/invoice-extractor/checkpoint-44
Removing /kaggle/working/invoice-extractor/checkpoint-308
Removing /kaggle/working/invoice-extractor/checkpoint-220
Removing /kaggle/working/invoice-extractor/checkpoint-176
Removing /kaggle/working/invoice-extractor/checkpoint-88
Removing /kaggle/working/invoice-extractor/checkpoint-440
Removing /kaggle/working/invoice-extractor/checkpoint-396
Removing /kaggle/working/invoice-extractor/checkpoint-352
4.0K	/kaggle/working/invoice-extractor
254M	/kaggle/working/invoice-extractor-final


In [14]:
!zip -r /kaggle/working/invoice_model.zip /kaggle/working/invoice-extractor-final

  adding: kaggle/working/invoice-extractor-final/ (stored 0%)
  adding: kaggle/working/invoice-extractor-final/training_args.bin (deflated 53%)
  adding: kaggle/working/invoice-extractor-final/config.json (deflated 52%)
  adding: kaggle/working/invoice-extractor-final/tokenizer.json (deflated 71%)
  adding: kaggle/working/invoice-extractor-final/model.safetensors (deflated 8%)
  adding: kaggle/working/invoice-extractor-final/tokenizer_config.json (deflated 42%)
